# Langflow Agent with memory, unboxed

This notebook runs the card **Langflow Agent with memory chat · local Qwen3.8-27B** one step per cell, with the card's own system prompt, its model on this machine, and its tools. Each cell prints what was sent and what came back; a folded line under it opens the code that ran.

| On the Langflow canvas | In this notebook |
|---|---|
| The Agent card calls the model and runs tools by itself | `agent.decide()` is one model call: its thinking, then the tool it chose. `agent.act()` runs that one tool. |
| A Run Flow card runs a whole flow | The flow's cards run one after another, each with its receipt |
| The Design pipeline is one tool call | One cell per card: start, memory, requirements, router, the writer, the Pilot, alignment, checks, router, trade-offs, review |
| Inside the pipeline, OpenCode writes design A and B in a sandbox | The **Langflow Agent, Pilot only** card writes them instead, driven here one turn per cell: `w.decide()` writes, `w.act()` is one official-Pilot call, the verdict goes back to the model |
| Write to memory shows Approve and Reject | `agent.approve(True)` or `agent.approve(False)` |

The writer is the one place this notebook departs from the canvas. The canvas's Design pipeline card hands A and B to OpenCode, an autonomous process whose own `./validate` calls cannot be paused between. Here the same work is given to the Langflow Agent, Pilot only card (project 2) with the same task text the pipeline hands every writer, so every Pilot round is a cell. On the Part 2 benchmark the two writers score the same tier (OpenCode 249/300, the Langflow agents 246/298 and 251/300). Everything before and after the writer is the same shell step the canvas calls, and the pipeline still re-parses what the writer hands in (④), as it does for OpenCode.

Two smaller differences: the knowledge-base card is left out (it searches Langflow's own vector store; Memory search reads the same cards), and the canvas does not light up. Google Colab cannot run this: the model, the shell service, the MCP server and the Pilot are on this machine.

## 0 · The machine

Start the services once, in a terminal: `python3 scripts/up.py --bg`. That starts the shell service, the MCP server, Ollama and Langflow. The results page may stay open.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bench" / "shell_service.py").is_file())
sys.path.insert(0, str(ROOT))
from notebooks.agent_walk import Agent, check_services

check_services()

# The writer's thinking switch. False: thinking off, the setting OpenCode uses for the 27B (faster, no reasoning text).
# None: send no switch, Ollama's default for this model (thinking on), the way Langflow's own card runs it.
WRITER_THINK = False

  ok  RTRC shell service                 http://127.0.0.1:8781 · results page open: True
  ok  RTRC MCP server                    http://127.0.0.1:8782/mcp · 9 tools
  ok  Ollama and the card's model        qwen3.8-27b-q3:64k installed · in GPU memory now
  ok  Official Pilot 0.62 (jar)          mvp/toolchain/kernel/sysml/jupyter-sysml-kernel-0.62.0-all.jar
  ok  OpenCode (pinned binary)           tools/opencode/opencode
  ok  Sandbox: bubblewrap                /usr/bin/bwrap
  ok  Sandbox network: slirp4netns       /usr/bin/slirp4netns

Everything the agent needs is up.


## 1 · The Agent card, unboxed

What the model receives with every call: the card's system prompt and its tools.

In [2]:
agent = Agent()
agent.show_card()


━━ The Agent card · Langflow Agent with memory chat · local Qwen3.8-27B · Langflow Agent · local Qwen3.8-27B ━━━━
Flow file        langflow/flows/4-langflow-agent-with-memory/langflow-agent-memory-chat-local.json
Model            qwen3.8-27b-q3:64k · provider Ollama · http://127.0.0.1:11434
Messages kept    30 · model calls per turn at most 20 · a failing tool's error goes back to the model: True
System prompt    3,871 characters (open it below)

Tools the model is given (9), as the canvas wires them to the Agent's Tools port:
   1. Memory-search_tool                             Run Flow · flow "Memory search" · Memory library → MiniLM + FAISS seeds → One hop of neighbours -> cards for the assistant
   2. Design-pipeline_tool                           Run Flow · flow "Design pipeline" · start → ① memory → ② requirements → new | adapt → ③ OpenCode designs → ④ Pilot → ⑤ align → ⑥ checks → trade-offs? → ⑦ review
   3. Validate-SysML-with-the-official-Pilot_tool    Run Flow · flow "Valida

## 2 · Turn 1 · the engineer asks

The engineer gives a need with all its numbers and asks the agent to look in memory first. The numbers are the brief confirmed in the Playground on 13 October.

Two separate cells follow: the model chooses, then the chosen tool runs.

In [3]:
agent.say("""I want a new design for this fixed-wing UAV. First check the memory for anything similar, then show me the exact brief you would send to the design pipeline.

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.""")


━━ Engineer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
I want a new design for this fixed-wing UAV. First check the memory for anything similar, then show me the exact brief you would send to the design pipeline.

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.


In [4]:
agent.decide()


━━ Step 1 · the model reads 2 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 user

 is

 asking

...

 tool

 calls

.


← 9.2 s · read 2496 tokens · wrote 919 tokens · stop: stop

The model chose (nothing has run yet; agent.act() runs it):
  Memory-search_tool({"flow_tweak_data": {"ChatInput-query~input_value": "fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio MaritimeISR FixedWingUav power storage upload"}})
  list_projects({})


{'content': '',
 'thinking': 'The user is asking me to do the following:\n1. First, check the memory for anything similar (search the memory)\n2. Next, present the exact brief that would be sent to the design pipeline (and wait for explicit confirmation before running)\n\nThe user has already provided a fairly complete brief containing all the numbers:\n- Fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure\n- Mid-360S lidar: 6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz\n- EO camera: 5 W\n- Search radar: 30 W\n- Edge computer: 14 W\n- Data radio: 10 W\n- Total power limit: 120 W\n- Lidar point cloud local recording: 25.6 Mbps, 8-hour retention, 256 GB storage\n- Processed data upload: 10 Mbps, uplink capacity 20 Mbps\n\nSo the brief format requirements are: vessel and its devices (with power consumption in W), power limit in W, local recording rate in Mbps, retention time (hours), storage in GB, upload rate in Mbps, uplink capacity in Mbps. All of these

In [5]:
agent.act()


━━ Step 2 · tool Memory-search_tool · canvas card “Tool · search memory (a flow) · Paper 2” ━━━━━━━━
arguments: {"flow_tweak_data": {"ChatInput-query~input_value": "fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio MaritimeISR FixedWingUav power storage upload"}}
The flow “Memory search” (langflow/flows/5-shared-tools/tool-memory-search.json) runs its 3 cards in order:

[1/3] Memory library: MaritimeISR + approved designs as cards
→ POST http://127.0.0.1:8781/tool/memory/library {}


← ok in 1.6 s
F0 memory: 196 cards in 4 source(s); 538 links
- Existing maritime model MaritimeISR: 63 cards (scenario/maritime-isr/MaritimeISR.sysml)
- Approved design 20261003T144529 · A: 44 cards (mvp/runs/20261015T034622-4db6ff79)
- Approved design 20261003T150116 · A: 45 cards (mvp/runs/20261015T034630-810fcaa8)
- Approved design 20261013T155413 · B: 44 cards (mvp/runs/20261019T070122-a00706c2)

Approved designs in memory: 3



[2/3] MiniLM + FAISS seeds: cosine >= 0.30 + exact-name bonus
→ POST http://127.0.0.1:8781/tool/memory/seeds {"query": "fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio MaritimeISR FixedWingUav power storage upload", "upstream": "F0 memory: 196 cards in 4 source(s); 538 links\n- Existing maritime model MaritimeISR: 63 cards (scenario/maritime-isr/MaritimeISR.sysml)\n…"}


← ok in 1.6 s
Seeds for "fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio Ma": 4 of 196 cards · sentence-transformers/all-MiniLM-L6-v2 · cosine ≥ 0.3 plus the exact-name bonus 0.1
- 0.79 · PartUsage MaritimeISR::FixedWingUav::radio  [MaritimeISR]
- 0.72 · PartUsage MaritimeISR::UnmannedSurfaceVessel::radio  [MaritimeISR]
- 0.69 · PartUsage MaritimeISR::uav  [MaritimeISR]
- 0.67 · PartDefinition MaritimeISR::Camera  [MaritimeISR]



[3/3] One hop of neighbours -> cards for the assistant
→ POST http://127.0.0.1:8781/tool/memory/hop {"query": "fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio MaritimeISR FixedWingUav power storage upload", "upstream": "Seeds for \"fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio Ma\": 4 of 196 cards · sentence-transformers/all-…"}


← ok in 1.6 s
One hop from the seeds: 3 neighbour(s); 7 cards in all · No related values in existing models: new-design path.
- via radio → PartDefinition MaritimeISR::FixedWingUav  [MaritimeISR]
- via radio → PartDefinition MaritimeISR::UnmannedSurfaceVessel  [MaritimeISR]
- via Camera → PortUsage MaritimeISR::Camera::imagery  [MaritimeISR]

Cards returned to the assistant:
- PartUsage MaritimeISR::FixedWingUav::radio
- PartUsage MaritimeISR::UnmannedSurfaceVessel::radio
- PartUsage MaritimeISR::uav
- PartDefinition MaritimeISR::Camera
- PartDefinition MaritimeISR::FixedWingUav
- PartDefinition MaritimeISR::UnmannedSurfaceVessel
- PortUsage MaritimeISR::Camera::imagery



Returned to the model as Memory-search_tool's result: 664 characters.


'One hop from the seeds: 3 neighbour(s); 7 cards in all · No related values in existing models: new-design path.\n- via radio → PartDefinition MaritimeISR::FixedWingUav  [MaritimeISR]\n- via radio → PartDefinition MaritimeISR::UnmannedSurfaceVessel  [MaritimeISR]\n- via Camera → PortUsage MaritimeISR::Camera::imagery  [MaritimeISR]\n\nCards returned to the assistant:\n- PartUsage MaritimeISR::FixedWingUav::radio\n- PartUsage MaritimeISR::UnmannedSurfaceVessel::radio\n- PartUsage MaritimeISR::uav\n- PartDefinition MaritimeISR::Camera\n- PartDefinition MaritimeISR::FixedWingUav\n- PartDefinition MaritimeISR::UnmannedSurfaceVessel\n- PortUsage MaritimeISR::Camera::imagery'

The rest of the turn: every further choice and tool, until the agent answers. It stops early if the agent calls the design pipeline or a tool that needs approval.

In [6]:
agent.run_until_reply()


━━ Step 3 · tool list_projects · canvas card “Tool · list projects” ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
arguments: {}
→ MCP tools/call list_projects {} · http://127.0.0.1:8782/mcp


← ok in 1.6 s

{
 "projects": [
  {
   "name": "MaritimeISR",
   "label": "Existing maritime model MaritimeISR",
   "cards": 63,
   "path": "scenario/maritime-isr/MaritimeISR.sysml"
  },
  {
   "name": "approved 20261003T144529·A",
   "label": "Approved design 20261003T144529 · A",
   "cards": 44,
   "path": "mvp/runs/20261015T034622-4db6ff79",
   "brief": "A harbor patrol unmanned surface vessel carries a visible camera drawing 28 W and an edge computer drawing 14 W. Add an infrared camera drawing 12 W. The vessel power limit is 60 W. Imagery is recorded locally at 12 Mbps for 8 hours on 256 GB of storage. Imagery is uploaded to shore at 12 Mbps over one 20 Mbps uplink.",
   "numbers": {
    "DesignA::vessel::uploadMbps": 12.0,
    "DesignA::vessel::visibleCamera::drawW": 28.0,
    "DesignA::vessel::edgeComputer::drawW": 14.0,
    "DesignA::vessel::storageGB": 256.0,
    "DesignA::vessel::powerLimitW": 60.0,
    "DesignA::vessel::infraredCamera::drawW": 12.0,
    "DesignA::vessel::ret


Returned to the model as list_projects's result: 3,479 characters.

━━ Step 4 · the model reads 5 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 user

...

1

0

 min

).


← 18.4 s · read 4225 tokens · wrote 1796 tokens · stop: stop

No tool call: the text after “answer ›” is the agent's reply, as the Playground shows it.


## 3 · Turn 2 · the engineer confirms; the agent calls the design pipeline

In [7]:
agent.say("Confirmed. Run the design pipeline with exactly that brief.")


━━ Engineer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Confirmed. Run the design pipeline with exactly that brief.


In [8]:
agent.decide()


━━ Step 5 · the model reads 7 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 engineer

 has

 confirmed

...

 I

 displayed

.


← 3.3 s · read 4807 tokens · wrote 297 tokens · stop: stop

The model chose (nothing has run yet; agent.act() runs it):
  Design-pipeline_tool({"flow_tweak_data": {"ChatInput-brief~input_value": "Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity."}})


{'content': '',
 'thinking': 'The engineer has confirmed the brief. I should run the design pipeline with the exact brief word for word. The brief contains an en dash "9–27 V DC". I have to send it exactly as it was confirmed. In the previous message I presented the brief with "9–27 V DC" (en dash). The engineer said "exactly that brief"—meaning the one I displayed. I\'ll use the version I displayed.\n',
 'tool_calls': [{'id': 'sQJvWqTVIbzM6VL0qLe4c4m2pedrCzTR',
   'function': {'index': 0,
    'name': 'Design-pipeline_tool',
    'arguments': {'flow_tweak_data': {'ChatInput-brief~input_value': 'Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256

The tool it chose is a flow of its own. This cell opens it, after running any other tool the model asked for first.

In [9]:
pipe = agent.until_pipeline()


━━ Step 6 · tool Design-pipeline_tool · canvas card “Tool · run the design pipeline (a flow) · Papers 2, 3, 1” ━━━━
arguments: {"flow_tweak_data": {"ChatInput-brief~input_value": "Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity."}}

This tool is a flow of its own: langflow/flows/5-shared-tools/tool-design-pipeline.json. Its cards run in this order; the next cells run them one by one.
   · RTRC Start (model qwen3.8-27b (Ollama))
   · ① Search memory · Paper 2
   · ② Requirements
   · Router: similar project found?
   · ③ New design, or ③ Adapt the si

### 3.1 RTRC Start

In [10]:
pipe.start()


━━ Design pipeline · RTRC Start ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/start {"brief": "Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ether…", "agent": "opencode", "model": "qwen3.8-27b (Ollama)", "replay": ""}
← ok in 0.0 s

run 20261021T073756 · live · agent opencode · qwen3.8-27b (Ollama)

Brief:

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.


{'ok': True,
 'run': '20261021T073756',
 'summary': 'run 20261021T073756 · live · agent opencode · qwen3.8-27b (Ollama)\n\nBrief:\n\nDesign a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.',
 'data': {'brief': 'Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded

### 3.2 ① Search memory · Paper 2

MiniLM + FAISS over the memory cards, seeds plus one hop. If an approved design is similar, the next card's text starts with “reuse”.

In [11]:
pipe.retrieve()


━━ Design pipeline · ① Search memory · Paper 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/retrieve {"run": "20261021T073756"}


← ok in 1.6 s

reuse · run 20261021T073756

① Memory: 196 cards; 6 retrieved (4 seeds, 2 one-hop neighbours); Reusable values found in existing models: reuse path.

Similar project: MaritimeISR (scenario/maritime-isr/MaritimeISR.sysml)
- seed radio · PartUsage · score 0.76 · MaritimeISR
- seed uav · PartUsage · score 0.71 · MaritimeISR
- seed downlinkMbps · AttributeUsage · score 0.71 · MaritimeISR
- seed radio · PartUsage · score 0.70 · MaritimeISR


{'ok': True,
 'run': '20261021T073756',
 'summary': 'reuse · run 20261021T073756\n\n① Memory: 196 cards; 6 retrieved (4 seeds, 2 one-hop neighbours); Reusable values found in existing models: reuse path.\n\nSimilar project: MaritimeISR (scenario/maritime-isr/MaritimeISR.sysml)\n- seed radio · PartUsage · score 0.76 · MaritimeISR\n- seed uav · PartUsage · score 0.71 · MaritimeISR\n- seed downlinkMbps · AttributeUsage · score 0.71 · MaritimeISR\n- seed radio · PartUsage · score 0.70 · MaritimeISR',
 'path': 'reuse',
 'data': {'path': 'reuse',
  'similar_project': {'name': 'MaritimeISR',
   'path': 'scenario/maritime-isr/MaritimeISR.sysml',
   'package': 'MaritimeISR',
   'brief': ''},
  'retrieval': {'method': 'hybrid',
   'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
   'threshold': 0.3,
   'name_bonus': 0.1,
   'index_key': '2cc0612cec2960ba',
   'sources': [{'name': 'MaritimeISR',
     'label': 'Existing maritime model MaritimeISR',
     'path': 'scenario/maritime-isr/M

### 3.3 ② Requirements

Numbered requirements. A number that is not in the brief blocks approval later.

In [12]:
pipe.requirements()


━━ Design pipeline · ② Requirements ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/requirements {"run": "20261021T073756"}


← ok in 6.6 s

reuse · run 20261021T073756 · similar project MaritimeISR

② 9 requirements, 1 questions; every number is in the brief
- R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit.
- R2: The Mid-360S lidar shall operate with a power draw of 6.5 W.
- R3: The EO camera shall operate with a power draw of 5 W.
- R4: The search radar shall operate with a power draw of 30 W.
- R5: The edge computer shall operate with a power draw of 14 W.
- R6: The data radio shall operate with a power draw of 10 W.
- R7: The lidar point cloud shall be recorded locally at a data rate of 25.6 Mbps.
- R8: The local storage shall provide a retention time of 8 hours for the recorded data.
- R9: The local storage shall have a capacity of 256 GB.
- question: The brief states processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity. Should I include requirements for upload rate (10 Mbps) and uplink capacity (20 Mbps) as well, or are the

{'ok': True,
 'run': '20261021T073756',
 'summary': 'reuse · run 20261021T073756 · similar project MaritimeISR\n\n② 9 requirements, 1 questions; every number is in the brief\n- R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit.\n- R2: The Mid-360S lidar shall operate with a power draw of 6.5 W.\n- R3: The EO camera shall operate with a power draw of 5 W.\n- R4: The search radar shall operate with a power draw of 30 W.\n- R5: The edge computer shall operate with a power draw of 14 W.\n- R6: The data radio shall operate with a power draw of 10 W.\n- R7: The lidar point cloud shall be recorded locally at a data rate of 25.6 Mbps.\n- R8: The local storage shall provide a retention time of 8 hours for the recorded data.\n- R9: The local storage shall have a capacity of 256 GB.\n- question: The brief states processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity. Should I include requirements for upload rate (10 Mbps)

### 3.4 Router: similar project found?

In [13]:
pipe.route_similar()


━━ Design pipeline · Router “Similar project found?” ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The router reads the requirements card's text and asks: does it start with “reuse”?
It starts with “reuse” → True: ③ Adapt the similar project.


'reuse'

### 3.5 ③ Design A · the Langflow Agent writes it, one Pilot round per cell

The writer gets the Pilot-only card's system prompt, the official Pilot as its one tool, and the task text the pipeline hands every writer (shown folded). Then:

- `wa.decide()` — one model call. The model writes SysML and asks for a Pilot call, or hands in.
- `wa.act()` — that one Pilot call: the Validate flow's two cards (save as a file, official Pilot). PASS or FAIL with the error lines goes back to the model.

Repeat the pair until the model hands in. The first cell below shows one round by hand; the next finishes the remaining rounds. On the local 27B a turn takes about a minute because the model thinks before it writes.

`WRITER_THINK` in the first cell sets whether the writer thinks before each turn. The outer agent keeps Langflow's default; `Agent(think=False)` would switch it off too. The run saved as `langflow_agent_with_memory-thinking-on.ipynb` was made with thinking on.

In [14]:
wa = pipe.writer("A", think=WRITER_THINK)


━━ Design pipeline · ③ design A · writer: Langflow Agent, Pilot only chat · Langflow Agent · qwen3.8-flash ━━━━
→ POST http://127.0.0.1:8781/run/prompt {"run": "20261021T073756", "key": "A", "adapt": true}
← ok in 0.0 s

run 20261021T073756

③ prompt for design A: adapt MaritimeISR · 5719 characters · 5 prompt parts

Card file        langflow/flows/2-coding-agents/langflow-agent-pilot-chat.json (on the canvas this card runs qwen3.8-flash; the pipeline sets the writer's model at run time, here qwen3.8-27b-q3:64k)
Model calls      at most 20 for this design (the card's limit)
Its one tool     Validate-SysML-with-the-official-Pilot_tool → flow “Validate SysML with the official Pilot” (langflow/flows/5-shared-tools/tool-validate-sysml.json): save the text as a file → official Pilot
Branch           adapt MaritimeISR


In [15]:
wa.decide()


━━ Design A · writer turn 1 · the model reads 2 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



← 8.2 s · read 2075 tokens · wrote 969 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 3142 characters · sha256 ffeb16e830ec


{'content': '',
 'thinking': '',
 'tool_calls': [{'id': 'Ha7tOGHyyxQXS8kTaI9ZPzOc3oIqxqmz',
   'function': {'index': 0,
    'name': 'Validate-SysML-with-the-official-Pilot_tool',
    'arguments': {'flow_tweak_data': {'ChatInput-sysml~input_value': 'package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port def DataPort;\n\n    part def Camera {\n        port imagery : ImageryPort;\n    }\n    part def Lidar {\n        port data : DataPort;\n    }\n    part def Radar {\n        port tracks : TrackPort;\n    }\n    part def Processor {\n        port imageryIn : ImageryPort;\n        port trackIn : TrackPort;\n        port dataIn : DataPort;\n        port uplink : LinkPort;\n    }\n    part def Radio {\n        port fromProcessor : LinkPort;\n        port toShore : LinkPort;\n    }\n\n    part def FixedWingUav {\n        part lidar : Lidar;\n        part eoCamera : Camera;\n        part searchRadar : Radar

In [16]:
wa.act()


━━ Design A · writer turn 2 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port …", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073813236867/DesignA.sysml

package: DesignA
- 100 line(s), 3141 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073813236867/DesignA.sysml\n\npackage: DesignA\n- 100 line(s), 3141 characters"}


← ok in 4.5 s
official Pilot PASS · package DesignA · run 20261021T073813-80831cb0
- line 44: Duplicate of inherited member name 'powerLimitW' from FixedWingUav
- line 45: Duplicate of inherited member name 'uplinkMbps' from FixedWingUav
- line 46: Duplicate of inherited member name 'uploadMbps' from FixedWingUav
- line 47: Duplicate of inherited member name 'storageGB' from FixedWingUav
- line 48: Duplicate of inherited member name 'dataRateMbps' from FixedWingUav
- line 49: Duplicate of inherited member name 'retentionHours' from FixedWingUav
- line 51: Duplicate of inherited member name 'lidar' from FixedWingUav
- line 54: Duplicate of inherited member name 'eoCamera' from FixedWingUav
- line 57: Duplicate of inherited member name 'searchRadar' from FixedWingUav
- line 60: Duplicate of inherited member name 'edgeComputer' from FixedWingUav
- line 63: Duplicate of inherited member name 'dataRadio' from FixedWingUav

Returned to the model as the tool's result: 916 characters. The mode

"official Pilot PASS · package DesignA · run 20261021T073813-80831cb0\n- line 44: Duplicate of inherited member name 'powerLimitW' from FixedWingUav\n- line 45: Duplicate of inherited member name 'uplinkMbps' from FixedWingUav\n- line 46: Duplicate of inherited member name 'uploadMbps' from FixedWingUav\n- line 47: Duplicate of inherited member name 'storageGB' from FixedWingUav\n- line 48: Duplicate of inherited member name 'dataRateMbps' from FixedWingUav\n- line 49: Duplicate of inherited member name 'retentionHours' from FixedWingUav\n- line 51: Duplicate of inherited member name 'lidar' from FixedWingUav\n- line 54: Duplicate of inherited member name 'eoCamera' from FixedWingUav\n- line 57: Duplicate of inherited member name 'searchRadar' from FixedWingUav\n- line 60: Duplicate of inherited member name 'edgeComputer' from FixedWingUav\n- line 63: Duplicate of inherited member name 'dataRadio' from FixedWingUav"

In [17]:
wa.decide()


━━ Design A · writer turn 3 · the model reads 4 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



← 6.0 s · read 3341 tokens · wrote 881 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2871 characters · sha256 86360eb4ffa5


{'content': '',
 'thinking': '',
 'tool_calls': [{'id': 'fc01uGfozyNaS130zKdlpCd3SuEUwjSz',
   'function': {'index': 0,
    'name': 'Validate-SysML-with-the-official-Pilot_tool',
    'arguments': {'flow_tweak_data': {'ChatInput-sysml~input_value': 'package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port def DataPort;\n\n    part def Camera {\n        port imagery : ImageryPort;\n    }\n    part def Lidar {\n        port data : DataPort;\n    }\n    part def Radar {\n        port tracks : TrackPort;\n    }\n    part def Processor {\n        port imageryIn : ImageryPort;\n        port trackIn : TrackPort;\n        port dataIn : DataPort;\n        port uplink : LinkPort;\n    }\n    part def Radio {\n        port fromProcessor : LinkPort;\n        port toShore : LinkPort;\n    }\n\n    part def FixedWingUav {\n        part lidar : Lidar;\n        part eoCamera : Camera;\n        part searchRadar : Radar

In [18]:
wa.act()


━━ Design A · writer turn 4 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port …", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073823800980/DesignA.sysml

package: DesignA
- 94 line(s), 2870 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073823800980/DesignA.sysml\n\npackage: DesignA\n- 94 line(s), 2870 characters"}


← ok in 4.3 s
official Pilot PASS · package DesignA · run 20261021T073823-7f24cf25
- line 45: Duplicate of inherited member name 'lidar' from FixedWingUav
- line 48: Duplicate of inherited member name 'eoCamera' from FixedWingUav
- line 51: Duplicate of inherited member name 'searchRadar' from FixedWingUav
- line 54: Duplicate of inherited member name 'edgeComputer' from FixedWingUav
- line 57: Duplicate of inherited member name 'dataRadio' from FixedWingUav

Returned to the model as the tool's result: 448 characters. The model's next turn decides whether to hand in.


"official Pilot PASS · package DesignA · run 20261021T073823-7f24cf25\n- line 45: Duplicate of inherited member name 'lidar' from FixedWingUav\n- line 48: Duplicate of inherited member name 'eoCamera' from FixedWingUav\n- line 51: Duplicate of inherited member name 'searchRadar' from FixedWingUav\n- line 54: Duplicate of inherited member name 'edgeComputer' from FixedWingUav\n- line 57: Duplicate of inherited member name 'dataRadio' from FixedWingUav"

Any rounds still needed, then the hand-in. `hand_in()` gives the reply to the pipeline exactly as a Langflow writer's reply would arrive; the pipeline parses it again in ④.

In [19]:
wa.run_until_reply()
wa.hand_in()


━━ Design A · writer turn 5 · the model reads 6 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



← 5.8 s · read 4383 tokens · wrote 836 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2682 characters · sha256 e25bdae789a2

━━ Design A · writer turn 6 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port …", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073833916635/DesignA.sysml

package: DesignA
- 87 line(s), 2681 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073833916635/DesignA.sysml\n\npackage: DesignA\n- 87 line(s), 2681 c

← ok in 4.3 s
official Pilot PASS · package DesignA · run 20261021T073833-cc8a4781

Returned to the model as the tool's result: 68 characters. The model's next turn decides whether to hand in.



━━ Design A · writer turn 7 · the model reads 8 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



answer › All

...

 powered

 device

).


← 7.5 s · read 5269 tokens · wrote 983 tokens · stop: stop

No tool call: this reply is the design the writer hands in. Run hand_in().

━━ Design A · hand in ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/written {"run": "20261021T073756", "key": "A", "text": "All checks pass. Here is the final package.\n\n```sysml\npackage DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n  …", "writer": "notebook: Langflow Agent, Pilot only", "model": "qwen3.8-27b (Ollama)", "tool_calls": "[{'n': 1, 'status': 'PASS', 'sha': 'ffeb16e830ec', 'run_id': '20261021T073813-80831cb0'}, {'n': 2, 'status': 'PASS', 'sha': '86360eb4ffa5…", "seconds": 40.6, "model_calls": 4}
← ok in 0.0 s

run 20261021T073756

③ Design A: 3480 characters written by notebook: Langflow Agent, Pilot only on qwen3.8-27b (Ollama); 3 Pilot call(s) by the writer; handed to ④

4 model call(s), 3 Pilot call(s) by the writer (3 PASS), 40.6 s. Hande

{'ok': True,
 'run': '20261021T073756',
 'summary': 'run 20261021T073756\n\n③ Design A: 3480 characters written by notebook: Langflow Agent, Pilot only on qwen3.8-27b (Ollama); 3 Pilot call(s) by the writer; handed to ④',
 'data': {'key': 'A', 'characters': 3480, 'written': ['A']},
 '_seconds': 0.0}

### 3.6 ③ Design B · the same, another arrangement

Design B is written new (no similar project to start from), so the writer has only the retrieved cards as examples. If it reaches the card's limit of 20 model calls, `hand_in()` hands in its last `DesignB` draft and says so; ④ then parses it and repairs on its errors with the pipeline's own model, the same second safety net OpenCode's hand-in gets.

In [20]:
wb = pipe.writer("B", think=WRITER_THINK)


━━ Design pipeline · ③ design B · writer: Langflow Agent, Pilot only chat · Langflow Agent · qwen3.8-flash ━━━━
→ POST http://127.0.0.1:8781/run/prompt {"run": "20261021T073756", "key": "B", "adapt": true}
← ok in 0.0 s

run 20261021T073756

③ prompt for design B: written new · 2911 characters · 5 prompt parts

Card file        langflow/flows/2-coding-agents/langflow-agent-pilot-chat.json (on the canvas this card runs qwen3.8-flash; the pipeline sets the writer's model at run time, here qwen3.8-27b-q3:64k)
Model calls      at most 20 for this design (the card's limit)
Its one tool     Validate-SysML-with-the-official-Pilot_tool → flow “Validate SysML with the official Pilot” (langflow/flows/5-shared-tools/tool-validate-sysml.json): save the text as a file → official Pilot
Branch           written new


In [21]:
wb.decide()


━━ Design B · writer turn 1 · the model reads 2 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



← 7.4 s · read 1324 tokens · wrote 870 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2668 characters · sha256 9b2997ef559d


{'content': '',
 'thinking': '',
 'tool_calls': [{'id': '4XDIrhvauURGWBa64bOsK6kdlZWXz5Ot',
   'function': {'index': 0,
    'name': 'Validate-SysML-with-the-official-Pilot_tool',
    'arguments': {'flow_tweak_data': {'ChatInput-sysml~input_value': 'package DesignB {\n\n  requirement R1 { doc "R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit." }\n  requirement R2 { doc "R2: The Mid-360S lidar shall operate with a power draw of 6.5 W." }\n  requirement R3 { doc "R3: The EO camera shall operate with a power draw of 5 W." }\n  requirement R4 { doc "R4: The search radar shall operate with a power draw of 30 W." }\n  requirement R5 { doc "R5: The edge computer shall operate with a power draw of 14 W." }\n  requirement R6 { doc "R6: The data radio shall operate with a power draw of 10 W." }\n  requirement R7 { doc "R7: The lidar point cloud shall be recorded locally at a data rate of 25.6 Mbps." }\n  requirement R8 { doc "R8: The local st

In [22]:
wb.act()


━━ Design B · writer turn 2 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignB {\n\n  requirement R1 { doc \"R1: The total power consumption of the UAV payload and avionics shall not exceed the system po…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073853099481/DesignB.sysml

package: DesignB
- 78 line(s), 2668 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073853099481/DesignB.sysml\n\npackage: DesignB\n- 78 line(s), 2668 characters"}


← ok in 3.3 s
official Pilot FAIL · package DesignB · run 20261021T073853-3863c84d
- line 3: no viable alternative at input 'doc'
- line 3: mismatched input '"R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit."' expecting RULE_REGULAR_COMMENT
- line 4: no viable alternative at input 'doc'
- line 4: mismatched input '"R2: The Mid-360S lidar shall operate with a power draw of 6.5 W."' expecting RULE_REGULAR_COMMENT
- line 5: no viable alternative at input 'doc'
- line 5: mismatched input '"R3: The EO camera shall operate with a power draw of 5 W."' expecting RULE_REGULAR_COMMENT
- line 6: no viable alternative at input 'doc'
- line 6: mismatched input '"R4: The search radar shall operate with a power draw of 30 W."' expecting RULE_REGULAR_COMMENT
- line 7: no viable alternative at input 'doc'
- line 7: mismatched input '"R5: The edge computer shall operate with a power draw of 14 W."' expecting RULE_REGULAR_COMMENT
- line 8: no viable

'official Pilot FAIL · package DesignB · run 20261021T073853-3863c84d\n- line 3: no viable alternative at input \'doc\'\n- line 3: mismatched input \'"R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit."\' expecting RULE_REGULAR_COMMENT\n- line 4: no viable alternative at input \'doc\'\n- line 4: mismatched input \'"R2: The Mid-360S lidar shall operate with a power draw of 6.5 W."\' expecting RULE_REGULAR_COMMENT\n- line 5: no viable alternative at input \'doc\'\n- line 5: mismatched input \'"R3: The EO camera shall operate with a power draw of 5 W."\' expecting RULE_REGULAR_COMMENT\n- line 6: no viable alternative at input \'doc\'\n- line 6: mismatched input \'"R4: The search radar shall operate with a power draw of 30 W."\' expecting RULE_REGULAR_COMMENT\n- line 7: no viable alternative at input \'doc\'\n- line 7: mismatched input \'"R5: The edge computer shall operate with a power draw of 14 W."\' expecting RULE_REGULAR_COMMENT\n-

In [23]:
wb.run_until_reply()
wb.hand_in()


━━ Design B · writer turn 3 · the model reads 4 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



← 5.3 s · read 2536 tokens · wrote 767 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2231 characters · sha256 fdc67b1c75cc

━━ Design B · writer turn 4 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignB {\n\n  /* R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit. */\n  req…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073901729961/DesignB.sysml

package: DesignB
- 72 line(s), 2231 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073901729961/DesignB.sysml\n\npackage: DesignB\n- 72 line(s), 2231 char

← ok in 3.0 s
official Pilot FAIL · package DesignB · run 20261021T073901-98bac486
- line 32: no viable alternative at input 'end'
- line 32: no viable alternative at input ';'
- line 36: no viable alternative at input 'end'
- line 36: no viable alternative at input ';'
- line 40: no viable alternative at input 'end'
- line 40: no viable alternative at input ';'
- line 44: no viable alternative at input 'end'
- line 44: no viable alternative at input ';'
- line 45: no viable alternative at input 'end'
- line 45: no viable alternative at input ';'
- line 46: no viable alternative at input 'end'
- line 46: no viable alternative at input ';'

Returned to the model as the tool's result: 632 characters. The model's next turn decides whether to fix and check again, or hand in.



━━ Design B · writer turn 5 · the model reads 6 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: False



← 5.2 s · read 3520 tokens · wrote 752 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2197 characters · sha256 76ec1bbbb42b

━━ Design B · writer turn 6 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignB {\n\n  /* R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit. */\n  req…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073909936812/DesignB.sysml

package: DesignB
- 72 line(s), 2197 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073909936812/DesignB.sysml\n\npackage: DesignB\n- 72 line(s), 2197 char

...

 carried

 over

.


← 7.4 s · read 5292 tokens · wrote 911 tokens · stop: stop

No tool call: this reply is the design the writer hands in. Run hand_in().

━━ Design B · hand in ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/written {"run": "20261021T073756", "key": "B", "text": "The package validated with the official SysML v2 Pilot (PASS). Here is the final design:\n\n```sysml\npackage DesignB {\n\n  /* R1: The total …", "writer": "notebook: Langflow Agent, Pilot only", "model": "qwen3.8-27b (Ollama)", "tool_calls": "[{'n': 1, 'status': 'FAIL', 'sha': '9b2997ef559d', 'run_id': '20261021T073853-3863c84d'}, {'n': 2, 'status': 'FAIL', 'sha': 'fdc67b1c75cc…", "seconds": 44.1, "model_calls": 5}
← ok in 0.0 s

run 20261021T073756

③ Design B: 3152 characters written by notebook: Langflow Agent, Pilot only on qwen3.8-27b (Ollama); 4 Pilot call(s) by the writer; handed to ④

5 model call(s), 4 Pilot call(s) by the writer (1 PASS), 44.1 s. Handed-

{'ok': True,
 'run': '20261021T073756',
 'summary': 'run 20261021T073756\n\n③ Design B: 3152 characters written by notebook: Langflow Agent, Pilot only on qwen3.8-27b (Ollama); 4 Pilot call(s) by the writer; handed to ④',
 'data': {'key': 'B', 'characters': 3152, 'written': ['A', 'B']},
 '_seconds': 0.0}

### 3.7 What the writer was given

The three read-only cards on the canvas: the rules, the attribute template, and the examples or the similar project.

In [24]:
pipe.prompt_cards()


━━ Design pipeline · ③ shown · Rules · Paper 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/tool/prompt-part {"run": "20261021T073756", "part": "rules"}
← ok in 0.0 s
run 20261021T073756 · Rules (design A, paper p3)

You write SysML v2 textual notation.

Rules:

- Output one fenced code block tagged sysml, and no prose before or after it.

- The file contains exactly one package whose name is given in the task.

- When you use Real, write `private import ScalarValues::*;` as the first line inside the package. In this toolchain an import inside a package needs the visibility keyword; a bare `import` is rejected. Do not import anything else.

- Use only names the task requires. Do not invent numeric limits.

- Do not change a number the task tells you to keep.

- The first line inside the package is exactly: private import ScalarValues::*;

- Declare attributes as `attribute name : Real = 12.0;` inside part usages, not part definitions, so values ar

### 3.8 Every Pilot call the writer made, A and B

In order, with the package each call was on: the writer may test small packages of its own between drafts, and those show up here as calls on other packages.

In [25]:
pipe.pilot_rounds()


━━ Design A · every Pilot call by the writer (3: 3 on DesignA, 0 on test packages), before the pipeline's own check ━━━━
Pilot call #1: package DesignA · PASS · sha256 ffeb16e830ec · Pilot run 20261021T073813-80831cb0
Pilot call #2: package DesignA · PASS · sha256 86360eb4ffa5 · Pilot run 20261021T073823-7f24cf25
Pilot call #3: package DesignA · PASS · sha256 e25bdae789a2 · Pilot run 20261021T073833-cc8a4781
handed in: 3480 characters · sha256 of its sysml block e25bdae789a2

━━ Design B · every Pilot call by the writer (4: 4 on DesignB, 0 on test packages), before the pipeline's own check ━━━━
Pilot call #1: package DesignB · FAIL · sha256 9b2997ef559d · Pilot run 20261021T073853-3863c84d
      line 3: no viable alternative at input 'doc'
      line 3: mismatched input '"R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit."' expecting RULE_REGULAR_COMMENT
      line 4: no viable alternative at input 'doc'
      line 4: mismatched in

### 3.9 ④ The pipeline checks again with the official Pilot

The pipeline does not take the writer's own Pilot calls as the verdict. It takes the ```sysml block of each reply and runs the official Pilot on it again; on a failure the complete error list goes back to the pipeline's model, up to 20 repairs. The cell also says whether the text it parsed is the text the writer's last Pilot call passed.

In [26]:
pipe.validate()


━━ Design pipeline · ④ Official SysML v2 Pilot ⇄ repair (the pipeline's own check) ━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/validate {"run": "20261021T073756"}


← ok in 9.7 s

run 20261021T073756

④ Official SysML v2 Pilot, with repair from errors (≤ 20)
- Design A: PASS after 1 parse(s), 0 repair(s)
- attempt 1: PASS
- Design B: PASS after 1 parse(s), 0 repair(s)
- attempt 1: PASS

Design A: the pipeline took the ```sysml block of the writer's reply and ran the official Pilot on it.
  attempt 1: PASS · Pilot run 20261021T073929-2811d4c4
  repairs after hand-in: 0
  The writer's last Pilot call on DesignA (3 on this design, 3 PASS) said PASS on text sha256 e25bdae789a2. The text the pipeline parsed (sha256 e25bdae789a2) is that same text.

Design B: the pipeline took the ```sysml block of the writer's reply and ran the official Pilot on it.
  attempt 1: PASS · Pilot run 20261021T073934-0070eb6f
  repairs after hand-in: 0
  The writer's last Pilot call on DesignB (4 on this design, 1 PASS) said PASS on text sha256 06ac2b1f14b0. The text the pipeline parsed (sha256 06ac2b1f14b0) is that same text.


{'ok': True,
 'run': '20261021T073756',
 'summary': 'run 20261021T073756\n\n④ Official SysML v2 Pilot, with repair from errors (≤ 20)\n- Design A: PASS after 1 parse(s), 0 repair(s)\n- attempt 1: PASS\n- Design B: PASS after 1 parse(s), 0 repair(s)\n- attempt 1: PASS',
 'data': [{'key': 'A',
   'sysml_status': 'PASS',
   'attempts': [{'attempt': 1,
     'status': 'PASS',
     'diagnostics': [],
     'sha': 'e25bdae789a2',
     'same_as_previous': False,
     'run_id': '20261021T073929-2811d4c4',
     'text': 'package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port def DataPort;\n\n    part def Camera {\n        port imagery : ImageryPort;\n    }\n    part def Lidar {\n        port data : DataPort;\n    }\n    part def Radar {\n        port tracks : TrackPort;\n    }\n    part def Processor {\n        port imageryIn : ImageryPort;\n        port trackIn : TrackPort;\n        port dataIn : DataPort;\n  

### 3.10 ⑤ Align with memory · Paper 1

In [27]:
pipe.align()


━━ Design pipeline · ⑤ Align with memory · Paper 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/align {"run": "20261021T073756"}


← ok in 10.5 s

run 20261021T073756

⑤ Alignment with MaritimeISR (paper 1): 4 pairs proposed, 4 with both names present, 4 declared; package AlignA_20261021T073756 official Pilot PASS
- DesignA::FixedWingUav → MaritimeISR::FixedWingUav · alias
- DesignA::uav → MaritimeISR::uav · allocation
- DesignA::FixedWingUav::dataRadio → MaritimeISR::FixedWingUav::radio · alias
- DesignA::FixedWingUav::uplinkMbps → MaritimeISR::FixedWingUav::downlinkMbps · alias
- Mappings are proposals. Code checks both names exist, writes alias and allocation statements from the two parses, then the official Pilot parses the alignment package. Neither source model is edited. A person accepts or rejects each pair.


{'ok': True,
 'run': '20261021T073756',
 'summary': 'run 20261021T073756\n\n⑤ Alignment with MaritimeISR (paper 1): 4 pairs proposed, 4 with both names present, 4 declared; package AlignA_20261021T073756 official Pilot PASS\n- DesignA::FixedWingUav → MaritimeISR::FixedWingUav · alias\n- DesignA::uav → MaritimeISR::uav · allocation\n- DesignA::FixedWingUav::dataRadio → MaritimeISR::FixedWingUav::radio · alias\n- DesignA::FixedWingUav::uplinkMbps → MaritimeISR::FixedWingUav::downlinkMbps · alias\n- Mappings are proposals. Code checks both names exist, writes alias and allocation statements from the two parses, then the official Pilot parses the alignment package. Neither source model is edited. A person accepts or rejects each pair.',
 'data': {'mappings': [{'design': 'DesignA::FixedWingUav',
    'existing': 'MaritimeISR::FixedWingUav',
    'reason': 'Both represent the fixed-wing UAV system architecture.',
    'design_exists': True,
    'existing_exists': True,
    'ok': True,
    'cons

### 3.11 ⑥ Checks and read-back

In [28]:
pipe.checks()


━━ Design pipeline · ⑥ Checks + read-back ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/checks {"run": "20261021T073756"}


← ok in 3.5 s

pass · run 20261021T073756

⑥ Checks on numbers read from the official parse; read-back (paper 3 query)
- Design A: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS
- read-back: missing 0, 9, 20, 27, 100, 120, 200
- Design B: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS
- read-back: missing 0, 9, 20, 27, 100, 120, 200
- eligible for approval: A, B


{'ok': True,
 'run': '20261021T073756',
 'summary': 'pass · run 20261021T073756\n\n⑥ Checks on numbers read from the official parse; read-back (paper 3 query)\n- Design A: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS\n- read-back: missing 0, 9, 20, 27, 100, 120, 200\n- Design B: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS\n- read-back: missing 0, 9, 20, 27, 100, 120, 200\n- eligible for approval: A, B',
 'verdict': 'pass',
 'data': {'failing': [],
  'eligible': ['A', 'B'],
  'designs': [{'key': 'A',
    'checks': [{'id': 'SOURCE',
      'label': 'Every design number is in the brief',
      'formula': 'each parsed literal appears in the brief or in a sourced requirement',
      'status': 'PASS',
      'actual': 11,
      'limit': 11,


### 3.12 Router: all checks pass?

In [29]:
pipe.route_checks()


━━ Design pipeline · Router “All four checks pass?” ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The router reads the checks card's text and asks: does it start with “pass”?
It starts with “pass” → True: straight to ⑦ Review.


True

### 3.13 Trade-offs (only when a check failed)

In [30]:
pipe.tradeoffs()


━━ Design pipeline · Trade-offs ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Skipped: the router sent the run straight to ⑦ Review because the checks passed.


{}

### 3.14 ⑦ Review

The flow's last card. Its text is what the Agent receives as the tool's result.

In [31]:
pipe.review()


━━ Design pipeline · ⑦ Diagram · Cameo export · review text ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/review {"run": "20261021T073756"}


← ok in 0.2 s

run 20261021T073756 · review for a person

Brief: Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.

Design A: official Pilot PASS after 1 parse(s); SOURCE PASS, POWER PASS, LINK PASS, STORE PASS, TRACE PASS, REQS UNKNOWN, MISSION PASS
- approval: eligible
- ⑦ diagram: http://127.0.0.1:8781/diagram?run=20261021T073929-2811d4c4&view=connections
- Cameo: export ready at analysis/cameo-exports/20261021T073756/DesignA.sysml (open it in Cameo's SysML v2 when RTRC provides the license)

Design B: official Pilot PASS after 1 parse(s); SOURCE P


This text is the flow's output (ChatOutput-review): Langflow hands it to the Agent as Design-pipeline_tool's result.


"run 20261021T073756 · review for a person\n\nBrief: Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.\n\nDesign A: official Pilot PASS after 1 parse(s); SOURCE PASS, POWER PASS, LINK PASS, STORE PASS, TRACE PASS, REQS UNKNOWN, MISSION PASS\n- approval: eligible\n- ⑦ diagram: http://127.0.0.1:8781/diagram?run=20261021T073929-2811d4c4&view=connections\n- Cameo: export ready at analysis/cameo-exports/20261021T073756/DesignA.sysml (open it in Cameo's SysML v2 when RTRC provides the license)\n\nDesign B: official Pilot PASS after 1 parse(s); SOURCE PASS, 

## 4 · The agent reads the review and answers

In [32]:
agent.give_back(pipe)


━━ Step 7 · the Design pipeline flow hands its last card's text back to the Agent ━━━━━━━━━━━━━━━━━━
1,467 characters (the review shown in ⑦) go back to the model as the tool's result.


In [33]:
agent.run_until_reply()


━━ Step 8 · the model reads 9 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 pipeline

 has

 completed

...

 would

 you

 like

?


← 6.0 s · read 5552 tokens · wrote 537 tokens · stop: stop

No tool call: the text after “answer ›” is the agent's reply, as the Playground shows it.


## 5 · Turn 3 · the approval card

In Langflow the Agent stops before Write to memory and shows Approve and Reject. Here `APPROVE` decides. It is `False`, so nothing is written; set it to `True` to store the design.

In [34]:
agent.say("Approve design B.")


━━ Engineer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Approve design B.


In [35]:
agent.run_until_reply()


━━ Step 9 · the model reads 11 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 engineer

 named

 Design

...

5

6

 B

".


← 1.9 s · read 5962 tokens · wrote 142 tokens · stop: stop

The model chose (nothing has run yet; agent.act() runs it):
  Write-to-memory_tool({"flow_tweak_data": {"ChatInput-approval~input_value": "20261021T073756 B"}})

━━ Step 10 · tool Write-to-memory_tool · canvas card “Tool · write to memory (a flow) · approval card · Paper 2” ━━━━
arguments: {"flow_tweak_data": {"ChatInput-approval~input_value": "20261021T073756 B"}}

Langflow stops here and shows “Tool execution requires approval” with Approve and Reject.
Nothing runs until agent.approve(True) (run it) or agent.approve(False) (refuse; the model is told).


In [36]:
APPROVE = False
agent.approve(APPROVE)


━━ Step 11 · the person refused Write-to-memory_tool ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Nothing ran. The model is told the person rejected the call.

Returned to the model as Write-to-memory_tool's result: 51 characters.


In [37]:
agent.run_until_reply()


━━ Step 12 · the model reads 13 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 write

 to

 memory

...

 before

 you

 decide

.


← 3.5 s · read 6049 tokens · wrote 316 tokens · stop: stop

No tool call: the text after “answer ›” is the agent's reply, as the Playground shows it.


## 6 · Check it yourself: the official Pilot on this run's files

In [38]:
pipe.recheck()


━━ Check it yourself · official Pilot on mvp/exam-inbox/discover-20261021T073756/DesignA.sysml ━━━━━
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port …", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T074005418662/DesignA.sysml

package: DesignA
- 87 line(s), 2681 characters
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T074005418662/DesignA.sysml\n\npackage: DesignA\n- 87 line(s), 2681 characters"}


← ok in 4.4 s
official Pilot PASS · package DesignA · run 20261021T074005-a95df071

━━ Check it yourself · official Pilot on mvp/exam-inbox/discover-20261021T073756/DesignB.sysml ━━━━━
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignB {\n\n  /* R1: The total power consumption of the UAV payload and avionics shall not exceed the system power limit. */\n  req…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T074009781983/DesignB.sysml

package: DesignB
- 72 line(s), 2120 characters
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T074009781983/DesignB.sysml\n\npackage: DesignB\n- 72 line(s), 2120 characters"}


← ok in 4.6 s
official Pilot PASS · package DesignB · run 20261021T074009-35a64e24


## 7 · The conversation as the Playground lists it, and where the files are

In [39]:
agent.transcript()
pipe.files()


━━ The conversation, as the Playground lists it ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Engineer › I want a new design for this fixed-wing UAV. First check the memory for anything similar, then show me the exact brief you would send to the design pipeline.

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.
  Agent called Memory-search_tool({"flow_tweak_data": {"ChatInput-query~input_value": "fixed-wing UAV Mid-360S lidar EO camera search radar edge computer data radio MaritimeISR FixedWingUav power storage upload"}})
  Agent called list

## Appendix · the canvas's writer, OpenCode

To see the writer the Design pipeline card actually uses, replace the cells of 3.5 and 3.6 with `pipe.design()`: OpenCode writes A and B in the sandbox and the cell prints each file it writes and each `./validate` it runs as they happen; `pipe.opencode("A")` then replays its whole session from its event log. Do not run both writers on one run. Run `20261021T063412` on results page 2.3 was made that way.